# Becke Partition 一阶梯度简单理解

In [1]:
from pyscf import gto, dft, lib, grad, hessian, data
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
def get_grids(xyz):
    mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()
    grids = dft.grid.Grids(mol).build()
    return mol, grids

## PySCF 解析与数值导数验证

需要留意，PySCF 先前使用的 `grids_response_cc` 似乎是由于后来引入 padding 的问题，其结果我不太确定是否正确。目前使用的是 PySCF 中用于计算 VV10 导数所用到的函数。

In [4]:
xyz_0 = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""
xyz_p = """
N  0.0001   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""
xyz_m = """
N -0.0001   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

In [5]:
mol, grids = get_grids(xyz_0)

一阶解析格点导数计算如下：

In [6]:
dw = hessian.rks.get_dweight_dA(mol, grids)
dw.shape

(4, 3, 43328)

作为例子，一阶数值导数的第一个分量计算如下：

In [7]:
tmp_num_d = (get_grids(xyz_p)[1].weights - get_grids(xyz_m)[1].weights) / (2 * 0.0001 / data.nist.BOHR)

解析与数值导数有强一致。

In [8]:
np.allclose(dw[0, 0], tmp_num_d)

True

## Becke Partition 一阶梯度实现与公式对应

## 记号与一阶导数总览

沿用 `10-1-becke_partition.ipynb` 的记号。格点 $g$ 的最终权重 (10-1 eq 22) 为

$$
w_g = w_g^{\mathrm{bare}}\,\frac{P_{A_g}(\mathbf r_g)}{\sum_B P_B(\mathbf r_g)}, \qquad A_g = \mathrm{atm\_idx}[g],
$$

其中 $w_g^{\mathrm{bare}}$ 为原始原子格点权重 (`grids.quadrature_weights`)，$P_A(\mathbf r_g)=\prod_{B\neq A}s_{AB}$ (10-1 eq 13)。记 $\Sigma_g=\sum_B P_B(\mathbf r_g)$。

**关键约定**：求导时格点坐标 $\mathbf r_g$ 视为固定。格点跟随原子运动的"格点响应"并不显式计算，而是通过平移不变性 (eq (8)) 归并到关联原子 $A_g$ 上——这与 PySCF 的 C 实现 `VXCbecke_weight_derivative` 及 Python 接口 `get_dweight_dA` 完全一致。

几何量记号：$\mathbf r_{Ag}=\mathbf R_A-\mathbf r_g$ (格点指向原子 $A$ 的向量)，$r_A=|\mathbf r_{Ag}|$；$\mathbf R_{AB}=\mathbf R_A-\mathbf R_B$，$R_{AB}=|\mathbf R_{AB}|$；$\mu_{AB}=(r_A-r_B)/R_{AB}$ (10-1 eq 11)。

对权重作商法则求导，得到本文的核心公式：

$$
\frac{\partial w_g}{\partial \mathbf R_G}
= w_g^{\mathrm{bare}}\left[\frac{1}{\Sigma_g}\frac{\partial P_{A_g}}{\partial \mathbf R_G}
- \frac{P_{A_g}}{\Sigma_g^2}\frac{\partial \Sigma_g}{\partial \mathbf R_G}\right].
\tag{1}
$$

注意 $\partial\Sigma_g/\partial\mathbf R_G$ 与关联原子 $A_g$ 无关 (它对所有 $P_B$ 求和)，可对所有格点统一计算；而 $\partial P_{A_g}/\partial\mathbf R_G$ 依赖 $A_g$。后续 eq (2)–(7) 给出这两项的表达式，eq (8) 给出 $G=A_g$ 的处理。

## $P_A$ 的对数导数

$P_A=\prod_{B\neq A}s_{AB}$ (10-1 eq 13)，其中 $s_{AB}=s_3(\nu_{AB})$ 仅通过 $\mu_{AB}$ 依赖原子坐标。由于格点固定，$\mu_{AB}=(r_A-r_B)/R_{AB}$ 只与端点原子 $\mathbf R_A,\mathbf R_B$ 有关。对乘积取对数导数，并引入对数导数核 $t_{AB}\equiv\frac{1}{s_{AB}}\frac{ds_{AB}}{d\mu_{AB}}$ (显式定义见 eq (5))：

$$
\frac{\partial P_A}{\partial \mathbf R_G}
= P_A\sum_{B\neq A}t_{AB}\,\frac{\partial \mu_{AB}}{\partial \mathbf R_G}.
\tag{2}
$$

求和中 $\partial\mu_{AB}/\partial\mathbf R_G$ 仅在 $G\in\{A,B\}$ 时非零 (见 eq (3))。这一对数导数形式正是 PySCF C 实现中 `switch_function_dmuds_over_s` 所计算的量，避免了把整个乘积 $P_A$ 直接对坐标差分的 $O(N_{\mathrm{atm}})$ 倍冗余。

## $\mu_{AB}$ 对原子坐标的导数

$\mu_{AB}=(r_A-r_B)/R_{AB}$，格点 $\mathbf r_g$ 固定。用到 $\partial r_A/\partial\mathbf R_A=\mathbf r_{Ag}/r_A$ (因 $r_A=|\mathbf r_g-\mathbf R_A|$)，以及 $\partial R_{AB}/\partial\mathbf R_A=\mathbf R_{AB}/R_{AB}$、$\partial R_{AB}/\partial\mathbf R_B=-\mathbf R_{AB}/R_{AB}$。对两个端点原子分别求导：

$$
\frac{\partial\mu_{AB}}{\partial\mathbf R_A}
= \frac{1}{R_{AB}}\left[\frac{\mathbf r_{Ag}}{r_A} - \mu_{AB}\frac{\mathbf R_{AB}}{R_{AB}}\right],
\tag{3a}
$$

$$
\frac{\partial\mu_{AB}}{\partial\mathbf R_B}
= \frac{1}{R_{AB}}\left[-\frac{\mathbf r_{Bg}}{r_B} + \mu_{AB}\frac{\mathbf R_{AB}}{R_{AB}}\right].
\tag{3b}
$$

对 $G\notin\{A,B\}$，$\partial\mu_{AB}/\partial\mathbf R_G=0$。式 (3a) 与 (3b) 中 $\mathbf R_{AB}$ 项符号相反，反映 $\mu_{AB}$ 对两端原子的反对称性；二者皆为 3 维向量 (xyz 分量)。后续把 $G$ 出现在乘积因子 $s_{XY}$ 中的两种角色记为：$G=X$ (A-角色，用 (3a)) 与 $G=Y$ (B-角色，用 (3b))。

## 开关函数 $s_{AB}$ 的导数与 $t_{AB}$

回顾 (10-1)：$\nu_{AB}=\mu_{AB}+a_{AB}(1-\mu_{AB}^2)$ (eq A2)，$p(\mu)=\frac32\mu-\frac12\mu^3$ (eq 19)，$f_3=p\circ p\circ p$ (eq 20)，$s_{AB}=\frac12(1-f_3(\nu_{AB}))$ (eq 21)。$p'(x)=\frac32(1-x^2)$。记 $f_1=p(\nu_{AB})$，$f_2=p(f_1)$，$f_3=p(f_2)$，$d\nu_{AB}/d\mu_{AB}=1-2a_{AB}\mu_{AB}$。由链式法则：

$$
\frac{ds_{AB}}{d\mu_{AB}} = -\frac12\,p'(f_2)\,p'(f_1)\,p'(\nu_{AB})\,\frac{d\nu_{AB}}{d\mu_{AB}}.
\tag{4}
$$

代入 eq (2) 中引用的对数导数核：

$$
t_{AB} = \frac{1}{s_{AB}}\frac{ds_{AB}}{d\mu_{AB}}.
\tag{5}
$$

**数值正则化**：当格点趋近 $B$ 原子时 $s_{AB}\to 0$、$t_{AB}\to\infty$，但乘积 $P_B\,t_{BG}$ 保持有限 ($P_B$ 含因子 $s_{BG}$)。为避免 $0\cdot\infty$，实现中对 $s_{AB}<10^{-14}$ 直接置 $t_{AB}=0$，与 PySCF C 实现的 `inv()` 一致；对常规格点该正则化几乎不触发。

## 组装 $\partial P_{A_g}/\partial\mathbf R_G$ 与 $\partial\Sigma/\partial\mathbf R_G$

对 $G\neq A_g$ (关联原子情形由 eq (8) 处理)，eq (2) 中仅 $B=G$ 的因子有贡献 ($G$ 取 B-角色)，用 (3b)：

$$
\frac{\partial P_{A_g}}{\partial \mathbf R_G}
= P_{A_g}\,t_{A_g G}\,\frac{\partial\mu_{A_g G}}{\partial\mathbf R_G}, \qquad G\neq A_g.
\tag{6}
$$

$\Sigma=\sum_B P_B$ 的导数按 $G$ 在各 $P_B$ 乘积中的两种角色拆分：$G$ 作为 A-角色出现在 $P_G$ 的每个因子 $s_{GC}$ ($C\neq G$，用 (3a))；$G$ 作为 B-角色出现在每个 $P_B$ ($B\neq G$) 的因子 $s_{BG}$ 中 (用 (3b))：

$$
\frac{\partial\Sigma}{\partial \mathbf R_G}
= \underbrace{P_G\sum_{C\neq G}t_{GC}\frac{\partial\mu_{GC}}{\partial\mathbf R_G}}_{\partial P_G/\partial\mathbf R_G\;(\text{eq 3a})}
\;+\; \underbrace{\sum_{B\neq G}P_B\,t_{BG}\,\frac{\partial\mu_{BG}}{\partial\mathbf R_G}}_{\text{eq 3b}}.
\tag{7}
$$

将 (6)、(7) 代入 (1) 即得 $G\neq A_g$ 的 $\partial w_g/\partial\mathbf R_G$。$\partial\Sigma/\partial\mathbf R_G$ 不依赖 $A_g$，可对所有格点统一向量化计算；$\partial P_{A_g}/\partial\mathbf R_G$ 则按每格点的 $A_g$ 取对应行。

## 平移不变性与效率

格点跟随原子运动，故权重只依赖相对位置，$\sum_G\partial w_g/\partial\mathbf R_G=0$。据此，关联原子 $A_g$ 的行无需显式求导，由剩余和给出：

$$
\frac{\partial w_g}{\partial\mathbf R_{A_g}} = -\sum_{G\neq A_g}\frac{\partial w_g}{\partial\mathbf R_G}.
\tag{8}
$$

这与 `get_dweight_dA` 末尾的 `dweight_dA[atm_idx, :, arange] = -sum(...)` 完全对应。padding 格点 (`atm_idx<0`) 权重为 0，导数亦为 0。

**效率**：$s_{AB},P_B,\Sigma$ 每格点一次 $O(N_{\mathrm{atm}}^2)$ 预计算；对每个导数原子 $G$，eq (7) 的两个求和与 eq (6) 均为 $O(N_{\mathrm{atm}})$ 并向量化于格点，总复杂度 $O(N_g\,N_{\mathrm{atm}}^2)$。相比 C 实现在每个 $G$ 内重算全部 $P_B$ 的 $O(N_g\,N_{\mathrm{atm}}^3)$，此处预计算 $P_B$ 消除了冗余 (代价是 $O(N_{\mathrm{atm}}^2)$ 的 $s_{AB}$ 存储，对常见体系可忽略)。

In [9]:
def becke_weight_derivative(grid_coords, grid_weights, atm_coords, a_factor, atm_idx):
    """核心求值函数 (纯 ndarray，对应 PySCF C 的 ``VXCbecke_weight_derivative``，但无 ctypes)。

    输入 (与 C 入参一一对应)：
        grid_coords  (N, 3)   格点 r_g (求导时固定)
        grid_weights (N,)     原始权重 w_g^bare  (C: grid_quadrature_weights)
        atm_coords   (M, 3)   原子坐标 R_A
        a_factor     (M, M)   radii 矫正表 a_{AB}
        atm_idx      (N,)     关联原子 A_g (int；<0 表示 padding)

    输出：
        dwdA (M, 3, N)  即 dw_g/dR_G，仅填充 G != A_g (及非 padding) 的行；
        G == A_g 的行留 0，交由接口层以平移不变性补齐 (eq (8))，与
        ``get_dweight_dA`` 中 C 调用后做 ``dweight_dA[atm_idx,...] = -sum(...)``
        的分工完全一致。

    公式 tag 见本 notebook 各 markdown 单元。
    """
    N = grid_coords.shape[0]
    M = atm_coords.shape[0]
    arangeN = np.arange(N)

    # --- 几何量 ---
    # r_{Ag} = R_A - r_g,  r_A = |r_{Ag}|
    rAg_vec = atm_coords[:, None, :] - grid_coords[None, :, :]      # (M, N, 3)
    rA = np.linalg.norm(rAg_vec, axis=2)                            # (M, N)
    rA_safe = np.where(rA > 1e-14, rA, 1.0)                         # inv() 风格安全除
    # R_{AB} = R_A - R_B
    Rab_vec = atm_coords[:, None, :] - atm_coords[None, :, :]       # (M, M, 3)
    Rab = np.linalg.norm(Rab_vec, axis=2)                            # (M, M)
    diag = np.eye(M, dtype=bool)
    Rab[diag] = 1.0                                                  # 避免 0/0
    Rab_inv = 1.0 / Rab
    Rab_inv[diag] = 0.0
    a_factor = np.array(a_factor, copy=True)
    a_factor[diag] = 0.0

    # eq (11): mu_{AB} = (r_A - r_B)/R_{AB} ;  eq (A2): nu_{AB} = mu + a (1 - mu^2)
    mu_AB = (rA[:, None, :] - rA[None, :, :]) * Rab_inv[:, :, None]   # (M, M, N)
    nu_AB = mu_AB + a_factor[:, :, None] * (1.0 - mu_AB ** 2)          # (M, M, N)

    # eq (19,20): f3 = p(p(p(nu))) ;  eq (21): s_{AB} = 1/2 (1 - f3)
    def p(x):
        return 1.5 * x - 0.5 * x ** 3                                  # eq (19)
    f1 = p(nu_AB)
    f2 = p(f1)
    f3 = p(f2)                                                          # eq (20)
    s_AB = 0.5 * (1.0 - f3)                                             # eq (21), (M, M, N)

    # eq (4): ds_{AB}/dmu_{AB} = -1/2 p'(f2) p'(f1) p'(nu) dnu/dmu,  p'(x)=3/2(1-x^2)
    p1 = 1.5 * (1.0 - nu_AB ** 2)
    p2 = 1.5 * (1.0 - f1 ** 2)
    p3 = 1.5 * (1.0 - f2 ** 2)
    dnu_dmu = 1.0 - 2.0 * a_factor[:, :, None] * mu_AB
    ds_dmu = -0.5 * p3 * p2 * p1 * dnu_dmu                             # (M, M, N)
    # eq (5): t_{AB} = (1/s) ds/dmu，配合 inv() 正则化 (s<1e-14 置 0)
    s_safe = np.where(s_AB > 1e-14, s_AB, 1.0)
    t_AB = np.where(s_AB > 1e-14, ds_dmu / s_safe, 0.0)

    # 排除自配对 (B=A)：eq (13) 乘积只对 B!=A
    s_AB[diag] = 1.0
    t_AB[diag] = 0.0

    # eq (13): P_A = prod_{B!=A} s_{AB} ;  Sigma = sum_B P_B
    P = np.prod(s_AB, axis=1)                                           # (M, N)
    Sigma = P.sum(axis=0)                                               # (N,)

    dwdA = np.zeros((M, 3, N))                                          # dwdA[G, xyz, g]

    for G in range(M):
        uG = rAg_vec[G] / rA_safe[G][:, None]                          # (N,3) r_{Gg}/r_G

        # ---- eq (7): dSigma/dR_G ----
        # (a) G 取 A-角色，因子 s_{GC} (C!=G)，eq (3a)
        mu_GC = mu_AB[G]                                                # (M, N) over C
        t_GC = t_AB[G]                                                  # (M, N)
        RabG = Rab[G]                                                   # (M,)
        RabG_vec = Rab_vec[G]                                           # (M, 3)
        dmu_GC = (Rab_inv[G][:, None, None]                             # (M, N, 3)
                  * (uG[None, :, :]
                     - mu_GC[:, :, None] * RabG_vec[:, None, :] / RabG[:, None, None]))
        dP_G = P[G][:, None] * np.einsum("cg,cgi->gi", t_GC, dmu_GC)   # dP_G/dR_G, (N,3)

        # (b) G 取 B-角色，因子 s_{BG} (B!=G)，eq (3b)
        mu_BG = mu_AB[:, G, :]                                          # (M, N) over B
        t_BG = t_AB[:, G, :]                                            # (M, N)
        RabBG = Rab[:, G]                                               # (M,)
        RabBG_vec = Rab_vec[:, G, :]                                    # (M, 3)
        dmu_BG = (Rab_inv[:, G][:, None, None]                          # (M, N, 3)
                  * (-uG[None, :, :]
                     + mu_BG[:, :, None] * RabBG_vec[:, None, :] / RabBG[:, None, None]))
        sumB = np.einsum("bg,bgi->gi", P * t_BG, dmu_BG)               # (N, 3)

        dSigma_dR_G = dP_G + sumB                                       # eq (7), (N, 3)

        # ---- eq (6): dP_{A_g}/dR_G  (G != A_g)，pair (A_g, G)，G 取 B-角色 ----
        dmu_AgG = dmu_BG[atm_idx, arangeN, :]                          # (N, 3) 选 B=A_g 行
        t_AgG = t_BG[atm_idx, arangeN]                                  # (N,)
        P_Ag = P[atm_idx, arangeN]                                      # (N,)
        dPA_dR_G = P_Ag[:, None] * t_AgG[:, None] * dmu_AgG            # eq (6), (N, 3)

        mask = (atm_idx != G) & (atm_idx >= 0)                          # G != A_g 且非 padding
        dPA_dR_G[~mask] = 0.0

        # ---- eq (1): dw_g/dR_G ----
        dw_G = grid_weights[:, None] * (
            dPA_dR_G / Sigma[:, None]
            - P_Ag[:, None] / Sigma[:, None] ** 2 * dSigma_dR_G
        )                                                               # (N, 3)
        dw_G[~mask] = 0.0
        dwdA[G] = dw_G.T                                                # (3, N)

    return dwdA


def becke_partition_dweight_dA(mol, grids):
    """接口函数 (对应 ``pyscf.hessian.rks.get_dweight_dA``)：从 mol/grids 抽取
    纯数组，调用核心函数，再以平移不变性补齐关联原子行。

    返回 dwdA[G, xyz, g]，形状 (natm, 3, ngrids)。
    """
    natm = mol.natm
    grid_coords = np.asarray(grids.coords, order="C")              # (N, 3)
    grid_weights = np.asarray(grids.quadrature_weights)            # (N,)
    atm_idx = np.asarray(grids.atm_idx)                            # (N,)
    atm_coords = np.asarray(mol.atom_coords(), order="C")         # (M, 3)

    # --- radii 矫正表 a_{AB} (与 get_dweight_dA 构造方式完全一致) ---
    radii_adjust = grids.radii_adjust
    atomic_radii = grids.atomic_radii
    if callable(radii_adjust) and atomic_radii is not None:
        f_adj = radii_adjust(mol, atomic_radii)
        a_factor = np.array([f_adj(i, j, 0)
                             for i in range(natm) for j in range(natm)]
                            ).reshape(natm, natm)
    else:
        a_factor = np.zeros((natm, natm))

    # 核心求值 (G != A_g 的行)
    dwdA = becke_weight_derivative(grid_coords, grid_weights, atm_coords,
                                   a_factor, atm_idx)

    # ---- eq (8): 平移不变性填充关联原子行 ----
    arangeN = np.arange(grids.coords.shape[0])
    dwdA[atm_idx, 0, arangeN] = -np.sum(dwdA[:, 0, :], axis=0)
    dwdA[atm_idx, 1, arangeN] = -np.sum(dwdA[:, 1, :], axis=0)
    dwdA[atm_idx, 2, arangeN] = -np.sum(dwdA[:, 2, :], axis=0)
    return dwdA

In [10]:
# 验证：与 PySCF get_dweight_dA (解析) 及数值导数比较
dw_ref = hessian.rks.get_dweight_dA(mol, grids)
dw_mine = becke_partition_dweight_dA(mol, grids)

print("shape:", dw_ref.shape, dw_mine.shape)
print("vs get_dweight_dA  max abs diff:", np.max(np.abs(dw_ref - dw_mine)))
print("vs get_dweight_dA  allclose   :", np.allclose(dw_ref, dw_mine))

# 数值导数 (沿用本 notebook 开头的 xyz_0/xyz_p/xyz_m，仅扰动 N 原子 x 方向)
num_d = (get_grids(xyz_p)[1].weights - get_grids(xyz_m)[1].weights) / (2 * 0.0001 / data.nist.BOHR)
print("vs numerical [0,0] allclose   :", np.allclose(dw_mine[0, 0], num_d))

shape: (4, 3, 43328) (4, 3, 43328)
vs get_dweight_dA  max abs diff: 2.1316282072803006e-14
vs get_dweight_dA  allclose   : True
vs numerical [0,0] allclose   : True
